# `JAXsim` Showcase: Parallel Simulation of a free-falling body

First, we install the necessary packages and import them.

In [ ]:
# @title Imports and setup
import os

# Deactivate GPU to avoid out of memory errors
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Set environment variable to avoid GPU out of memory errors
%env XLA_PYTHON_CLIENT_MEM_PREALLOCATE=false

import time

import jax
import jax.numpy as jnp
import rod
from rod.builder.primitives import SphereBuilder, BoxBuilder

import jaxsim.typing as jtp
from jaxsim import logging
from jaxsim.api.common import VelRepr

from jaxsim.mujoco import (
    MujocoVideoRecorder,
    MujocoModelHelper,
    RodModelToMjcf,
    SdfToMjcf,
    UrdfToMjcf,
)

logging.set_logging_level(logging.LoggingLevel.INFO)
logging.info(f"Running on {jax.devices()}")

We will use a simple sphere model to simulate a free-falling body. The spheres set will be composed of 9 spheres, each with a different position. The spheres will be simulated in parallel, and the simulation will be run for 3000 steps corresponding to 3 seconds of simulation.

**Note**: Parallel simulations are independent of each other, the different position is imposed only to show the parallelization visually.

In [ ]:
# @title Create a sphere model
rod_model = (
    BoxBuilder(x=0.30, y=0.30, z=1.0, mass=1.0, name="box")
    .build_model()
    .add_link()
    .add_inertial()
    .add_visual()
    .add_collision()
    .build()
)

rod_model.add_frame(
    rod.Frame(
        name="contact_frame",
        attached_to="box_link",
        pose=rod.Pose(pose=[-0.15, -0.15, -0.5, 0.0, 0.0, 0.0], relative_to="box_link"),
    )
)
model_sdf_string = rod.Sdf(version="1.10", model=rod_model).serialize(pretty=True)

In [ ]:
model_sdf_string

JAXsim offers a simple functional API in order to interact in a memory-efficient way with the simulation. Four main elements are used to define a simulation:

- `model`: an object that defines the dynamics of the system.
- `data`: an object that contains the state of the system.
- `integrator`: an object that defines the integration method.
- `integrator_state`: an object that contains the state of the integrator.

In [ ]:
import jaxsim.api as js
from jaxsim import integrators
import jaxopt
from jaxsim.terrain import FlatTerrain

dt = 0.001
integration_time = int(1.0 / dt)

model = js.model.JaxSimModel.build_from_model_description(
    model_description=model_sdf_string,
    terrain=FlatTerrain.build(height=-10.0),
)

from jaxsim.rbda.contacts.soft import SoftContactsParams

data = js.data.JaxSimModelData.build(
    model=model,
    contacts_params=SoftContactsParams.build(0.0, 0.0, 0.0),
    velocity_representation=VelRepr.Mixed,
)
integrator = integrators.fixed_step.RungeKutta4SO3.build(
    dynamics=js.ode.wrap_system_dynamics_for_integration(
        model=model,
        data=data,
        system_dynamics=js.ode.system_dynamics,
    ),
)
# with jax.disable_jit():
integrator_state = integrator.init(x0=data.state, t0=0.0, dt=dt)

In [ ]:
mcjf_string, assets = UrdfToMjcf.convert(urdf=model_sdf_string)
mj_helper = MujocoModelHelper.build_from_xml(
    mjcf_description=mcjf_string, assets=assets
)
recorder = MujocoVideoRecorder(
    model=mj_helper.model, data=mj_helper.data, fps=int(1 / dt), width=640, height=480
)

It is possible to automatically choose a good set of parameters for the terrain. 

By default, in JaxSim a sphere primitive has 250 collision points. This can be modified by setting the `JAXSIM_COLLISION_SPHERE_POINTS` environment variable.

Given that at its steady-state the sphere will act on two or three points, we can estimate the ground parameters by explicitly setting the number of active points to these values.

Let's create a position vector for a 3x3 grid. Every sphere will be placed at a different height.

In [ ]:
import numpy as np

# Primary Calculations
envs_per_row = 1  # @slider(2, 10, 1)
initial_height = 0.5
env_spacing = 0.5
edge_len = env_spacing * (2 * envs_per_row - 1)


# Create Grid
def grid(edge_len, envs_per_row):
    edge = jnp.linspace(-edge_len, edge_len, envs_per_row)
    xx, yy = jnp.meshgrid(edge, edge)

    poses = [
        [[xx[i, j], yy[i, j], initial_height], [0, 0, 0]]
        for i in range(xx.shape[0])
        for j in range(yy.shape[0])
    ]

    return jnp.array(poses)


logging.info(f"Simulating {envs_per_row**2} environments")
poses = grid(edge_len, envs_per_row)

In order to parallelize the simulation, we first need to define a function `simulate` for a single element of the batch.

In [ ]:
jit = True


# Define a function to simulate a single model instance
def simulate(
    data: js.data.JaxSimModelData, integrator_state: dict, pose: jnp.array
) -> tuple:

    def compute_force(model, data, position):
        with data.switch_velocity_representation(VelRepr.Inertial):
            J = js.frame.jacobian(
                model=model,
                data=data,
                frame_index=contact_frame,
                output_vel_repr=VelRepr.Mixed,
            )[:3]

        τ = jnp.zeros_like(data.joint_positions())
        τ_constraints = jnp.zeros_like(data.joint_positions())

        S = jnp.block([jnp.zeros(shape=(model.dofs(), 6)), jnp.eye(model.dofs())]).T

        with data.switch_velocity_representation(VelRepr.Mixed):
            h = js.model.free_floating_bias_forces(model=model, data=data)

        # R = jnp.diag(r)
        with data.switch_velocity_representation(VelRepr.Mixed):
            J_dot = js.frame.jacobian_derivative(
                model=model,
                data=data,
                frame_index=contact_frame,
                output_vel_repr=VelRepr.Mixed,
            )[:3]

        # Compute the smooth contact force.
        qf_smooth = S @ (jnp.atleast_1d(τ - τ_constraints)) - h

        with data.switch_velocity_representation(VelRepr.Mixed):
            M_inv = jnp.linalg.inv(
                js.model.free_floating_mass_matrix(model=model, data=data)
            )
        with data.switch_velocity_representation(VelRepr.Mixed):
            ν = data.generalized_velocity()

        # Calculate quantities for the linear optimization problem.
        A = J @ M_inv @ J.T
        b = J @ M_inv @ qf_smooth + J_dot @ ν

        objective = lambda x: jnp.sum(0.5 * (A @ x + b) ** 2)

        # Compute the 3D linear force in C[W] frame
        opt = jaxopt.ProjectedGradient(
            fun=objective,
            projection=jaxopt.projection.projection_non_negative,
            maxiter=100,
            implicit_diff=False,
            maxls=20,
            verbose=0,
        )

        CW_f_Ci = opt.run(init_params=jnp.zeros_like(b)).params.reshape(-1, 3).squeeze()
        CW_f_Ci6D = jnp.zeros(shape=(6,)).at[:3].set(CW_f_Ci)

        W_H_CW = (
            js.frame.transform(model=model, data=data, frame_index=contact_frame)
            .at[0:3, 0:3]
            .set(jnp.eye(3))
        )

        W_f_Ci6D = js.common.ModelDataWithVelocityRepresentation.other_representation_to_inertial(
            array=CW_f_Ci6D,
            other_representation=VelRepr.Mixed,
            transform=W_H_CW,
            is_force=True,
        )

        W_f_Li_terrain = W_f_Ci6D

        return W_f_Li_terrain.squeeze()

    data = data.reset_base_position(base_position=pose)
    x_t_i = []
    joint_positions = []
    forces = []
    link_positions = []
    base_orientations = []
    in_contact = []
    Jd_nu = []

    base = model.link_names().index("box_link")
    contact_frame = js.frame.name_to_idx(model=model, frame_name="contact_frame")

    get_collidable_points_pos_vel = lambda data: js.frame.transform(
        model=model, data=data, frame_index=contact_frame
    )[:3, 3]

    compute_force_jit = lambda data, position: compute_force(
        model=model,
        data=data,
        position=position,
    )

    compute_force_jit = jax.jit(compute_force_jit) if jit else compute_force_jit

    try:

        for _ in range(integration_time):
            with data.switch_velocity_representation(VelRepr.Mixed):

                W_o_C = get_collidable_points_pos_vel(data)

                references = js.references.JaxSimModelReferences.build(
                    model=model,
                    data=data,
                )

                with references.switch_velocity_representation(VelRepr.Inertial):

                    W_f_F = compute_force_jit(data, W_o_C)

                    references = references.apply_frame_forces(
                        forces=W_f_F,
                        model=model,
                        data=data,
                        frame_names=model.frame_names()[0],
                        additive=False,
                    )

                with (
                    data.switch_velocity_representation(VelRepr.Inertial),
                    references.switch_velocity_representation(VelRepr.Inertial),
                ):

                    data, integrator_state = js.model.step(
                        dt=dt,
                        model=model,
                        data=data,
                        integrator=integrator,
                        integrator_state=integrator_state,
                        joint_forces=None,
                        link_forces=(
                            F := references.link_forces(
                                model=model, data=data
                            ).squeeze()
                        ),
                    )

            x_t_i.append(data.base_position())
            joint_positions.append(data.joint_positions())
            base_orientations.append(data.base_orientation())
            forces.append(F.squeeze())
            link_positions.append(
                js.link.transform(model=model, data=data, link_index=base)[3, :3]
            )
            in_contact.append(np.array(W_o_C)[2] < 0)
            Jd_nu.append(
                js.frame.jacobian_derivative(model, data, frame_index=contact_frame)[:3]
                @ data.generalized_velocity()
            )

            print(f"Simulated time: {_ * dt:.3f}s", end="\r")
        # finally:
        return (
            x_t_i,
            base_orientations,
            joint_positions,
            forces,
            link_positions,
            in_contact,
            Jd_nu,
        )
    except Exception as e:
        raise e

We will make use of `jax.vmap` to simulate multiple models in parallel. This is a very powerful feature of JAX that allows to write code that is very similar to the single-model case, but can be executed in parallel on multiple models.
In order to do so, we need to first apply `jax.vmap` to the `simulate` function, and then call the resulting function with the batch of different poses as input.

Note that in our case we are vectorizing over the `pose` argument of the function `simulate`, this correspond to the value assigned to the `in_axes` parameter of `jax.vmap`:

`in_axes=(None, None, 0)` means that the first two arguments of `simulate` are not vectorized, while the third argument is vectorized over the zero-th dimension.

In [ ]:
# Run and time the simulation
now = time.perf_counter()

# x_t = simulate_vectorized(data, integrator_state, poses[:, 0]).
x_t, base_orientations, joint_positions, forces, link_positions, in_contact, Jd_nu = (
    simulate(data, integrator_state, poses[:, 0])
)

comp_time = time.perf_counter() - now

logging.info(
    f"Running simulation with {envs_per_row**2} models took {comp_time} seconds."
)
logging.info(
    f"This corresponds to an RTF (Real Time Factor) of {(envs_per_row**2 * integration_time / 1000 / comp_time):.2f}"
)

In [ ]:
print(f"{model.link_names()=}")

In [ ]:
(np.array(Jd_nu) == 0).all()

In [ ]:
np.array(Jd_nu).reshape(1500, -1).shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

in_contact = np.array(in_contact)

plt.plot(
    np.arange(len(in_contact[:])) * dt,
    in_contact[:],
    label="In contact",
)
# plt.plot(
#     np.arange(len(x_t[:])) * dt, np.array(Jd_nu).reshape(1500, 24)[:], label="JDnu"
# )  # label=[f"JD_nu_{component}" for component in range(24)])

plt.plot(np.arange(len(x_t[:])) * dt, np.array(x_t)[:, 2])
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Height [m]")
plt.title("Trajectory of the model's base")
plt.legend()
plt.show()

In [ ]:
from pathlib import Path

for pose, orientation, s_t in zip(x_t, base_orientations, joint_positions, strict=True):
    mj_helper.set_base_position(pose)
    mj_helper.set_base_orientation(orientation)
    # mj_helper.set_joint_positions(positions=s_t, joint_names=model.joint_names())
    recorder.record_frame()  # camera_name="pendulum_camera")

import datetime

import mediapy as media

media.show_video(recorder.frames, fps=1 / dt)

recorder.write_video(
    path=(
        save_path := Path.cwd()
        / Path(
            f"{model.name()}_{data.velocity_representation}_{datetime.datetime.now().isoformat()}.mp4"
        )
    ),
    exist_ok=True,
)

In [ ]:
save_path

Now let's extract the data from the simulation and plot it. We expect to see the height time series of each sphere starting from a different value.

In [ ]:
forces = np.array([force for force in forces])
link_positions = np.array(link_positions)

In [ ]:
forces

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(
    np.arange(len(link_positions)) * dt,
    link_positions,
    # label=["L", "R"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Link CoM")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

in_contact = np.array(in_contact)

# plt.plot(
#     np.arange(len(in_contact[:])) * dt,
#     in_contact.any(axis=1)[:],
#     label="In contact",
# )
plt.plot(
    np.arange(len(forces[:])) * dt,
    forces[:],
    label=["X", "Y", "Z", "Rx", "Ry", "Rz"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Force [N]")
plt.title("Contact forces")
plt.legend()
plt.show()